# 4.10 · KNN 回归 / K-Nearest Neighbors Regression

> **课程定位 / Where this fits**
> **Part 4 第 10 课, 进入"基于实例"的非参数模型**。前 9 课都是参数模型(学一组权重)。KNN 是极致的**惰性/非参数**模型: **不训练**, 预测时直接看最近 k 个邻居的平均。最简单的非线性回归, 也是理解"维度诅咒"和"局部 vs 全局"的最佳案例。
> The laziest non-parametric model: no training, just average the k nearest neighbors at prediction time.

> 💡 **面试相关 / Interview-relevant**
> - "KNN 怎么工作, 为什么叫惰性学习" ★★★★
> - "k 怎么选, 大小 k 的影响" ★★★★（偏差-方差）
> - "维度诅咒怎么害 KNN" ★★★★★
> - "KNN 为什么必须缩放" ★★★★（距离, 3.4）

---

## 学习目标 / Learning Objectives
1. 理解 KNN 的**惰性预测**机制 + 从零实现。
2. **k = 偏差-方差旋钮**: k 小过拟合, k 大欠拟合。
3. **距离加权** vs 均匀平均。
4. **维度诅咒**为何让 KNN 在高维失效。
5. 复习"距离模型必须缩放"(3.4)。

## 目录 / TOC
1. [惰性学习: 预测时才工作 ⭐](#1)
2. [💎 数据 + 从零实现](#2)
3. [k 是偏差-方差旋钮 ⭐](#3)
4. [距离加权](#4)
5. [⚠ 维度诅咒 ⭐](#5)
6. [对照 sklearn + 缩放](#6)
7. [小结](#7)


<a id="1"></a>
## 1. 惰性学习: 预测时才工作 ⭐ / Lazy Learning

**KNN 没有"训练"**。它只是**记住全部训练数据**。预测一个新点 $\mathbf{x}$ 时：
1. 算 $\mathbf{x}$ 到所有训练点的距离
2. 找最近的 $k$ 个
3. 回归: 返回这 $k$ 个邻居的 **y 平均值**

$$\hat{y}(\mathbf{x}) = \frac{1}{k}\sum_{i \in N_k(\mathbf{x})} y_i$$

| | 参数模型 (线性/SVR) | KNN (惰性/非参数) |
|---|---|---|
| 训练 | 学权重(慢) | 只存数据(秒) |
| 预测 | 算 $\mathbf{x}^\top\mathbf{w}$(快) | 算所有距离(**慢** $O(n)$/点) |
| 模型形式 | 固定函数 | **无显式形式**, 数据即模型 |
| 假设 | 有(如线性) | **几乎无** — 让数据自己说话 |

**惰性 (lazy)**: 把所有计算推迟到预测时。**非参数 (non-parametric)**: 参数量随数据增长(存全部数据), 没有固定数量的权重。
Lazy: defers all computation to prediction. Non-parametric: "parameters" grow with data (it stores everything).


In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.preprocessing import StandardScaler
from sklearn.model_selection import train_test_split, cross_val_score
sns.set_theme(style="whitegrid")
rng = np.random.default_rng(42)

# Diamonds: 用数值特征预测价格 / predict diamond price
df = sns.load_dataset("diamonds").sample(5000, random_state=0).reset_index(drop=True)
X = df[["carat","depth","table","x","y","z"]].values
y = df["price"].values
print(f"Diamonds: {X.shape}, 预测价格")
print(df[["carat","price"]].describe().round(2).T)


<a id="2"></a>
## 2. 从零实现 / From Scratch


In [ ]:
# 从零 KNN 回归 (1D 演示更直观) / from-scratch KNN on 1D for clarity
def knn_predict(X_train, y_train, X_query, k=5):
    preds = []
    for xq in X_query:
        dists = np.sqrt(((X_train - xq)**2).sum(axis=1))   # 欧氏距离
        nn = np.argsort(dists)[:k]                          # 最近 k 个
        preds.append(y_train[nn].mean())                    # 邻居 y 平均
    return np.array(preds)

# 1D: carat → price / 1-feature demo
xc = df["carat"].values.reshape(-1, 1)
order = np.argsort(xc.ravel())
xc, yc = xc[order], y[order]
x_plot = np.linspace(xc.min(), xc.max(), 200).reshape(-1, 1)

from sklearn.neighbors import KNeighborsRegressor
sk_pred = KNeighborsRegressor(n_neighbors=5).fit(xc, yc).predict(x_plot)
my_pred = knn_predict(xc, yc, x_plot, k=5)
print(f"从零 vs sklearn 最大差异: {np.abs(my_pred - sk_pred).max():.4f} → 一致")

fig, ax = plt.subplots(figsize=(7, 4))
ax.scatter(xc, yc, alpha=0.15, s=8, label="数据")
ax.plot(x_plot, my_pred, "r-", lw=2, label="KNN (k=5)")
ax.set_xlabel("carat"); ax.set_ylabel("price"); ax.legend()
ax.set_title("KNN 回归: 阶梯状曲线 (每段=局部邻居平均)")
plt.tight_layout(); plt.show()
print("KNN 曲线呈阶梯/锯齿状 — 因为是局部平均, 没有平滑的函数形式")


<a id="3"></a>
## 3. k 是偏差-方差旋钮 ⭐ / k as the Bias-Variance Knob

$k$ 是 KNN 的核心超参（4.3 阶数、4.4 λ 的同类）：

| k | 行为 | 偏差-方差 |
|---|---|---|
| **k=1** | 完全跟随最近点 | 极低偏差, **极高方差**(过拟合, 追噪声) |
| **k 适中** | 局部平滑 | 平衡 |
| **k 大** | 大范围平均 | **高偏差**(欠拟合), 低方差; k=n 时退化成预测全局均值 |


In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(15, 4))
for ax, k in zip(axes, [1, 20, 500]):
    pred = KNeighborsRegressor(n_neighbors=k).fit(xc, yc).predict(x_plot)
    ax.scatter(xc, yc, alpha=0.1, s=6)
    ax.plot(x_plot, pred, "r-", lw=2)
    tag = "过拟合(追噪声)" if k==1 else ("刚好" if k==20 else "欠拟合(过平滑)")
    ax.set_title(f"k={k}  {tag}")
plt.tight_layout(); plt.show()

# CV 选 k / select k via CV
Xc_full = df[["carat"]].values
ks = [1, 3, 5, 10, 20, 50, 100, 200]
cv = [cross_val_score(KNeighborsRegressor(n_neighbors=k), Xc_full, y, cv=5, scoring="r2").mean() for k in ks]
best_k = ks[np.argmax(cv)]
print(f"CV 各 k 的 R²: {dict(zip(ks, np.round(cv,3)))}")
print(f"最优 k = {best_k}")
print("k=1 过拟合(锯齿), k 太大欠拟合(平线); CV 选中等 k")


<a id="4"></a>
## 4. 距离加权 / Distance Weighting

均匀 KNN 让 k 个邻居**同等投票**——但近的邻居应该更可信。**距离加权**: 权重 $\propto 1/\text{距离}$, 近邻影响更大。


In [ ]:
Xc_full = df[["carat"]].values
uniform = cross_val_score(KNeighborsRegressor(n_neighbors=20, weights="uniform"), Xc_full, y, cv=5, scoring="r2").mean()
distance = cross_val_score(KNeighborsRegressor(n_neighbors=20, weights="distance"), Xc_full, y, cv=5, scoring="r2").mean()
print(f"均匀权重 (uniform): CV R² = {uniform:.3f}")
print(f"距离加权 (distance): CV R² = {distance:.3f}")
print("距离加权让近邻影响更大 → 通常略好, 且预测曲线更平滑(近邻主导)")

# 可视化平滑差异 / visualize smoothness
fig, ax = plt.subplots(figsize=(7, 3.5))
for w, c in [("uniform","orange"),("distance","red")]:
    p = KNeighborsRegressor(n_neighbors=20, weights=w).fit(xc, yc).predict(x_plot)
    ax.plot(x_plot, p, color=c, lw=2, label=f"weights={w}")
ax.scatter(xc, yc, alpha=0.08, s=6); ax.legend(); ax.set_title("距离加权 vs 均匀 (k=20)")
plt.tight_layout(); plt.show()


<a id="5"></a>
## 5. ⚠ 维度诅咒 ⭐ / The Curse of Dimensionality

**KNN 的致命弱点**。高维空间里, **所有点都变得几乎等距** → "最近邻"失去意义 → KNN 失效。

**直觉**: 维度越高, 体积越集中在"边缘壳层", 任意两点的距离趋于相同。下面实测: 随维度增加, 最近邻距离 / 最远邻距离 → 1（即近邻和远邻没区别了）。
In high dimensions all points become nearly equidistant — "nearest" loses meaning.


In [ ]:
# 实测维度诅咒: 最近/最远邻距离比 → 1 / nearest-to-farthest ratio -> 1
dims = [1, 2, 5, 10, 50, 100, 500]
ratios = []
for d in dims:
    pts = rng.uniform(0, 1, (1000, d))
    q = rng.uniform(0, 1, (1, d))
    dists = np.sqrt(((pts - q)**2).sum(axis=1))
    ratios.append(dists.min() / dists.max())     # 越接近1越糟

fig, ax = plt.subplots(figsize=(7, 4))
ax.plot(dims, ratios, "o-", lw=2)
ax.set_xscale("log"); ax.set_xlabel("维度 d"); ax.set_ylabel("最近邻距离 / 最远邻距离")
ax.set_title("维度诅咒: 高维下近邻和远邻距离趋同 → KNN 失效")
plt.tight_layout(); plt.show()
print(f"{'维度':<6} {'近/远距离比':<12}")
for d, r in zip(dims, ratios):
    print(f"{d:<6} {r:<12.3f}")
print("\nd=1: 近邻比远邻近很多(比值小, KNN 有效)")
print("d=500: 近邻≈远邻(比值≈1, '最近邻'毫无意义) → KNN 在高维彻底失效")
print("→ 高维必须先降维(PCA, 0.7) 或改用树模型/线性模型")


<a id="6"></a>
## 6. 对照 sklearn + 缩放 / sklearn & Scaling

KNN 用距离 → **必须标准化**（3.4 节实测过, KNN 不缩放灾难）。多特征时尤其关键。


In [ ]:
from sklearn.pipeline import make_pipeline

X_tr, X_te, y_tr, y_te = train_test_split(X, y, test_size=0.3, random_state=0)

# 不缩放 vs 缩放 / unscaled vs scaled
raw = KNeighborsRegressor(n_neighbors=10).fit(X_tr, y_tr).score(X_te, y_te)
scaled = make_pipeline(StandardScaler(), KNeighborsRegressor(n_neighbors=10)).fit(X_tr, y_tr).score(X_te, y_te)
print(f"KNN 不缩放: test R² = {raw:.3f}")
print(f"KNN 缩放后: test R² = {scaled:.3f}")
print("缩放是 KNN 的必需品 (3.4): carat∈[0,5] 但 price 相关的 x,y,z∈[0,10], 不缩放则大量纲特征主导距离")


<a id="7"></a>
## 7. 小结 / Summary

```
KNN: 惰性(不训练只存数据) + 非参数(无固定权重数)
预测: 最近 k 个邻居的 y 平均 (回归); 阶梯状曲线
k = 偏差-方差旋钮: k=1 过拟合(锯齿), k 大欠拟合(平), CV 选中等
距离加权: 近邻影响更大, 更平滑
维度诅咒 ⭐: 高维下所有点近似等距 → "最近邻"失意义 → KNN 失效
必须标准化 (3.4); 预测慢 O(n)/点 (大数据用 KD-tree/Ball-tree 加速)
```

### 💡 面试速查
1. **惰性学习**: 不训练, 预测时算所有距离取 k 邻居平均
2. **k 是偏差-方差旋钮**: 小过拟合大欠拟合
3. **维度诅咒**: 高维近邻≈远邻 → KNN 死 → 先降维
4. **KNN 必须缩放** (距离, 3.4)
5. **预测慢** O(n); KD-tree 加速低维

### 下一节
**4.11 决策树回归**——KNN 是"局部平均", 树是"递归切分空间再局部平均"。树天生抗维度诅咒、不需缩放, 是随机森林/GBDT 的基石。
